# FieldAid Walkthrough Demo
## Gemma 4 Good Hackathon

This notebook clones the full repository (including walkthrough media), sets up Ollama with Gemma 4 E4B, starts the server, and launches the walkthrough demo.

### Step 1: Clone Repository and Install

In [ ]:
!git clone --branch wt/walkthrough --depth 1 https://github.com/Kushalk0677/fieldaid-offline-disaster-copilot.git
%cd fieldaid-offline-disaster-copilot
print("Cloned repo with walkthrough folder.")

In [ ]:
!pip install -r requirements.txt -q

import sys
print(f"Python {sys.version}")

from app.main import app
print("FastAPI app loaded successfully.")

### Step 2: Install Ollama and Pull Gemma 4 E4B (~9.6 GB)
Leave this running. Takes 5-10 minutes.

In [ ]:
!apt-get update -qq && apt-get install -y -qq zstd curl > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh

import os, subprocess, time

os.system("killall -9 ollama 2>/dev/null || true")
print("Starting Ollama server...")
subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(3)

print("Pulling gemma4:e4b (~9.6 GB). Stand by...")
!ollama pull gemma4:e4b

print("\nVerification:")
!ollama run gemma4:e4b "Say hello in one word." --quiet

### Step 3: Configure and Start Server

In [ ]:
import os
from pathlib import Path

os.environ["FIELDAID_RUNTIME_CACHE"] = str(Path.home() / ".fieldaid_runtime")
os.environ["PYTHONPATH"] = "."

Path("data/uploads").mkdir(parents=True, exist_ok=True)
Path(os.environ["FIELDAID_RUNTIME_CACHE"]).mkdir(parents=True, exist_ok=True)

print("Runtime configured.")

classifier = Path("models/fieldaid-medic-image-classifier-final/best.pt")
yolo = Path("yolo11n.pt")
for name, p in [("Classifier", classifier), ("YOLO", yolo)]:
    if p.exists():
        print(f"  {name}: found ({p.stat().st_size / 1e6:.1f} MB)")
    else:
        print(f"  {name}: missing")

wt_media = Path("walkthrough/media")
main_media = Path("media")
if wt_media.exists():
    files = [f.name for f in wt_media.iterdir()]
    print(f"  Walkthrough media: {len(files)} files")
else:
    print("  Walkthrough media: not found")

import shutil

wt_static = Path("walkthrough/static")
main_static = Path("static")
if wt_static.exists():
    for f in wt_static.iterdir():
        dest = main_static / f.name
        shutil.copy2(str(f), str(dest))
        print(f"  Copied {f.name} -> static/")
else:
    print("  Walkthrough static: not found")

wt_app = Path("walkthrough/app")
main_app = Path("app")
if wt_app.exists():
    for f in wt_app.iterdir():
        if f.is_file():
            dest = main_app / f.name
            shutil.copy2(str(f), str(dest))
            print(f"  Copied {f.name} -> app/")
else:
    print("  Walkthrough app: not found")

if wt_media.exists():
    main_media.mkdir(exist_ok=True)
    for f in wt_media.iterdir():
        dest = main_media / f.name
        shutil.copy2(str(f), str(dest))
        print(f"  Copied {f.name} -> media/")
else:
    print("  Walkthrough media: not found")

wt_data = Path("walkthrough/data")
main_data = Path("data")
if wt_data.exists():
    for item in wt_data.rglob("*"):
        if item.is_file():
            dest = main_data / item.relative_to(wt_data)
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(str(item), str(dest))
    print("  Copied data/ files")
else:
    print("  Walkthrough data: not found")

wt_models = Path("walkthrough/models")
main_models = Path("models")
if wt_models.exists():
    for item in wt_models.rglob("*"):
        if item.is_file():
            dest = main_models / item.relative_to(wt_models)
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(str(item), str(dest))
    print("  Copied models/ files")
else:
    print("  Walkthrough models: not found")

print("Walkthrough overlay applied.")

In [ ]:
import threading, time

def run_server():
    os.system("uvicorn app.main:app --host 0.0.0.0 --port 8000 > /tmp/fieldaid.log 2>&1")

print("Starting FieldAid server on port 8000...")
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(4)

import httpx
try:
    resp = httpx.get("http://localhost:8000", timeout=5)
    print(f"Server running (status {resp.status_code})")
    print("\nClick the preview panel above or visit http://localhost:8000")
    print("Then click Start Walkthrough button in the bottom-right corner.")
except Exception as e:
    print(f"Server not ready yet: {e} (try this cell again)")

### Step 4: Walkthrough Instructions

1. Click the **preview panel** above (Colab will open it)
2. On the webpage, click **Start Walkthrough** (bottom-right corner, pulses green)
3. Click **Next** to advance through each step
   - Forms auto-fill with scenario data
   - Images and videos auto-load from walkthrough media
   - Gemma 4 E4B runs locally for each analysis
4. Use **Previous** to review past steps
5. Click **End** (top-right X) when finished

The walkthrough runs 11 steps covering shelter intake, damage assessment, video scanning, trust gates, dashboard, incidents, Discord handoff, and sync export.